# 8. Edit a query: `replaceClause()` and `removeSubQuery()`

Once a query is built, you don't have to throw it out and rebuild from scratch to change a filter. Two helpers let you transform an existing query into a new one:

- `picsure::replaceClause(query, target, replacement)` swaps every structural match of `target` for `replacement`.
- `picsure::removeSubQuery(query, target)` deletes every structural match of `target`. Empty groups left behind are pruned automatically; an empty result query raises an error.

Both functions are **non-mutating** — they return a new query handle. The original is untouched, so you can fork variants from a common base.

In [ ]:
library(picsure)

In [ ]:
open_hpds_session <- picsure::connect(
  platform = picsure::Platform$BDC_OPEN
)

## Find the two AGE variables we'll swap between

We pick one continuous age variable from each of two studies so we can demonstrate replacing one for the other in a query.

In [ ]:
facets <- picsure::facets(open_hpds_session)
picsure::addFacet(facets, "dataset_id", c("phs000810", "phs000007"))

In [ ]:
results <- picsure::searchDictionary(open_hpds_session, "age", facets = facets)
results

In [ ]:
# phs000007 -- used in the initial query; later swapped out
age5_phs000007 <- results[results$display == "age5" & results$name == "phv00177938", ]
age5_phs000007_clause <- picsure::createSubQuery(
  age5_phs000007$conceptPath[[1]],
  type = picsure::ClauseType$FILTER,
  min  = 30,
  max  = 40
)

# phs000810 -- the replacement
age_immi_phs000810 <- results[results$display == "AGE_IMMI", ]
age_immi_phs000810_clause <- picsure::createSubQuery(
  age_immi_phs000810$conceptPath[[1]],
  type = picsure::ClauseType$FILTER,
  min  = 30,
  max  = 40
)

## Add a sex variable from FHS

In [ ]:
fhs_facet <- picsure::facets(open_hpds_session)
picsure::addFacet(fhs_facet, "dataset_id", "phs000007")

fhs_sex_results <- picsure::searchDictionary(open_hpds_session, "phv00253990", facets = fhs_facet)
fhs_sex_results

## Build the base query: (Female AND age5 30-40) OR (Male AND age5 30-40)

This is the same nested OR-of-ANDs shape from notebook 4.

In [ ]:
fhs_sex_male_clause <- picsure::createSubQuery(
  fhs_sex_results$conceptPath[[1]],
  type       = picsure::ClauseType$FILTER,
  categories = list("Male")
)
fhs_male_and_30_to_40 <- picsure::buildQuery(
  list(fhs_sex_male_clause, age5_phs000007_clause),
  operator = picsure::GroupOperator$AND
)

fhs_sex_female_clause <- picsure::createSubQuery(
  fhs_sex_results$conceptPath[[1]],
  type       = picsure::ClauseType$FILTER,
  categories = list("Female")
)
fhs_female_and_30_to_40 <- picsure::buildQuery(
  list(fhs_sex_female_clause, age5_phs000007_clause),
  operator = picsure::GroupOperator$AND
)

fhs_female_or_male_30_to_40 <- picsure::buildQuery(
  list(fhs_female_and_30_to_40, fhs_male_and_30_to_40),
  operator = picsure::GroupOperator$OR
)

fhs_female_or_male_30_to_40$to_query_json()

In [ ]:
picsure::runQuery(open_hpds_session, fhs_female_or_male_30_to_40)
# Verified with UI: 77 +-3

## `replaceClause()` -- swap age5 (phs000007) for AGE_IMMI (phs000810)

Every occurrence of `age5_phs000007_clause` inside the query tree (both AND branches, here) is replaced with `age_immi_phs000810_clause`. The original `fhs_female_or_male_30_to_40` is not modified.

In [ ]:
swapped <- picsure::replaceClause(
  fhs_female_or_male_30_to_40,
  age5_phs000007_clause,
  age_immi_phs000810_clause
)
swapped$to_query_json()

In [ ]:
picsure::runQuery(open_hpds_session, swapped)
# AGE_IMMI is sparser than age5; expect a much smaller cohort
# (typically below the small-cohort obfuscation threshold)

## `removeSubQuery()` -- drop the age filter entirely

Removing the AGE_IMMI clause from `swapped` leaves the two AND groups holding only their sex clauses. The result is effectively `Female OR Male` for FHS participants.

In [ ]:
fhs_female_or_male <- picsure::removeSubQuery(swapped, age_immi_phs000810_clause)
fhs_female_or_male$to_query_json()

In [ ]:
picsure::runQuery(open_hpds_session, fhs_female_or_male)
# Verified with UI: ~1267 +-3